# Notebook 06 - Deep Learning Model (TensorFlow)
**Proyek:** Pulsevera - Predict, Prevent, Prevail  
**Role:** AI Engineer (Fathan & Shafira)  
**Tujuan:** Membangun model deep learning dengan TensorFlow **Functional API**, menerapkan dua komponen kustom (custom loss = Focal Loss + custom callback = EarlyStoppingByRecall), serta mencatat training dengan TensorBoard.

**Komponen yang memenuhi requirement Main Quest:**
1. Functional API (bukan Sequential)
2. Custom Loss — `focal_loss` (untuk class imbalance ~5.6%)
3. Custom Callback — `EarlyStoppingByRecall` (stop bila recall positif stagnan)
4. Class weighting otomatis berdasarkan distribusi label
5. Format model produksi: `.keras` (sesuai requirement Coding Camp)

## 1. Setup & Import

In [ ]:
import json
import os
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings('ignore')

BASE_DIR = Path('..')
FINAL_DIR = BASE_DIR / 'data' / 'final'
MODELS_DIR = BASE_DIR / 'ml-api' / 'models'
FIGURES_DIR = BASE_DIR / 'notebooks' / 'figures'
LOG_DIR = BASE_DIR / 'ml-api' / 'tensorboard_logs'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available    : {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Load Data + Scaling

Untuk neural network, **wajib scaling**. Pakai scaler yang sama dengan ML supaya FastAPI konsisten.

In [ ]:
X_train = pd.read_csv(FINAL_DIR / 'X_train.csv')
X_test = pd.read_csv(FINAL_DIR / 'X_test.csv')
y_train = pd.read_csv(FINAL_DIR / 'y_train.csv').squeeze().values
y_test = pd.read_csv(FINAL_DIR / 'y_test.csv').squeeze().values
FEATURE_ORDER = X_train.columns.tolist()

scaler_path = MODELS_DIR / 'scaler.pkl'
if scaler_path.exists():
    scaler = joblib.load(scaler_path)
    print(f'Reuse scaler dari {scaler_path}')
else:
    scaler = StandardScaler().fit(X_train)
    joblib.dump(scaler, scaler_path)
    print(f'Scaler baru dibuat dan disimpan ke {scaler_path}')

X_train_s = scaler.transform(X_train).astype('float32')
X_test_s = scaler.transform(X_test).astype('float32')
y_train = y_train.astype('float32')
y_test = y_test.astype('float32')

print(f'X_train_s: {X_train_s.shape}   X_test_s: {X_test_s.shape}')
print(f'Positif train: {y_train.mean()*100:.2f}%   Positif test: {y_test.mean()*100:.2f}%')

## 3. Custom Loss — Focal Loss

Focal loss menurunkan kontribusi sample yang sudah "mudah" diklasifikasi (umumnya kelas negatif yang dominan) dan fokus ke sample positif yang sulit. Sangat cocok untuk class imbalance parah seperti dataset ini.

Formula: $FL(p_t) = -\alpha (1 - p_t)^\gamma \log(p_t)$

In [ ]:
@keras.saving.register_keras_serializable(package='pulsevera', name='focal_loss')
def focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25):
    """Focal loss — fokuskan training ke sampel positif yang sulit diprediksi."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
    bce = -(y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
    p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
    alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
    modulating = tf.pow(1.0 - p_t, gamma)
    return tf.reduce_mean(alpha_t * modulating * bce)

y_demo = tf.constant([[0.0], [1.0], [0.0], [1.0]])
p_demo = tf.constant([[0.1], [0.9], [0.4], [0.2]])
print('focal_loss demo:', focal_loss(y_demo, p_demo).numpy())

## 4. Custom Callback — EarlyStoppingByRecall

Stop training bila recall validasi (kelas positif) tidak meningkat dalam beberapa epoch. Selain itu, simpan bobot terbaik berdasarkan recall, bukan val_loss — sesuai prioritas medical screening.

In [ ]:
class EarlyStoppingByRecall(keras.callbacks.Callback):
    """Stop training bila val_recall stagnan + restore bobot terbaik."""

    def __init__(self, patience=5, min_delta=1e-3, monitor='val_recall', verbose=1):
        super().__init__()
        self.patience = patience
        self.min_delta = min_delta
        self.monitor = monitor
        self.verbose = verbose
        self.best = -np.inf
        self.wait = 0
        self.best_epoch = 0
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current = logs.get(self.monitor)
        if current is None:
            return
        if current - self.best > self.min_delta:
            self.best = current
            self.wait = 0
            self.best_epoch = epoch
            self.best_weights = self.model.get_weights()
            if self.verbose:
                print(f'  [EarlyStoppingByRecall] new best {self.monitor}={current:.4f} @ epoch {epoch+1}')
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
                if self.best_weights is not None:
                    self.model.set_weights(self.best_weights)
                if self.verbose:
                    print(f'  [EarlyStoppingByRecall] stop @ epoch {epoch+1} — best {self.monitor}={self.best:.4f} (epoch {self.best_epoch+1}); restoring weights.')


## 5. Bangun Model — Functional API

Arsitektur: Dense 256 → BN → Dropout → Dense 128 → BN → Dropout → Dense 64 → Dropout → Dense 1 (sigmoid).

In [ ]:
def build_heart_disease_model(input_dim, dropout_rate=0.3):
    """Pulsevera DL classifier — TensorFlow Functional API."""
    inputs = keras.Input(shape=(input_dim,), name='lifestyle_input')
    x = layers.Dense(256, activation='relu', kernel_initializer='he_normal')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(128, activation='relu', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(64, activation='relu', kernel_initializer='he_normal')(x)
    x = layers.Dropout(dropout_rate / 2)(x)

    outputs = layers.Dense(1, activation='sigmoid', name='heart_attack_risk')(x)
    return keras.Model(inputs=inputs, outputs=outputs, name='pulsevera_dl_model')

model = build_heart_disease_model(input_dim=X_train_s.shape[1])
model.summary()



## 6. Compile dengan Custom Loss + Class Weight

In [ ]:
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight = {0: float(cw[0]), 1: float(cw[1])}
print('Class weights:', class_weight)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=focal_loss,
    metrics=[
        keras.metrics.BinaryAccuracy(name='accuracy'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.AUC(name='auc'),
    ],
)

## 7. Training + TensorBoard Logging

In [ ]:
run_name = datetime.now().strftime('run_%Y%m%d_%H%M%S')
run_dir = LOG_DIR / run_name

callbacks = [
    EarlyStoppingByRecall(patience=5, monitor='val_recall', verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    keras.callbacks.TensorBoard(log_dir=str(run_dir), histogram_freq=1),
]

history = model.fit(
    X_train_s, y_train,
    validation_split=0.15,
    epochs=40,
    batch_size=512,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=2,
)

print(f'\nTensorBoard log: {run_dir}')
print(f'Jalankan lokal: tensorboard --logdir {LOG_DIR}')

## 8. Visualisasi Training History

In [ ]:
h = history.history
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (name, train_key, val_key) in zip(
    axes.flatten(),
    [('Loss', 'loss', 'val_loss'), ('Accuracy', 'accuracy', 'val_accuracy'),
     ('Recall', 'recall', 'val_recall'), ('AUC', 'auc', 'val_auc')]
):
    if train_key in h:
        ax.plot(h[train_key], label='train', linewidth=2)
    if val_key in h:
        ax.plot(h[val_key], label='val', linewidth=2)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('epoch'); ax.set_ylabel(name.lower()); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Training History — Pulsevera DL', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dl_training_history.png', dpi=150)
plt.show()

## 9. Evaluasi pada Test Set

In [ ]:
y_proba_dl = model.predict(X_test_s, batch_size=1024, verbose=0).ravel()

for thr in [0.3, 0.4, 0.5, 0.6]:
    y_pred = (y_proba_dl >= thr).astype(int)
    print(f'Threshold {thr}: '
          f'acc={accuracy_score(y_test, y_pred):.4f} '
          f'precision(1)={precision_score(y_test, y_pred, pos_label=1, zero_division=0):.4f} '
          f'recall(1)={recall_score(y_test, y_pred, pos_label=1, zero_division=0):.4f} '
          f'f1(1)={f1_score(y_test, y_pred, pos_label=1, zero_division=0):.4f}')

best_thr = max(
    [0.3, 0.35, 0.4, 0.45, 0.5],
    key=lambda t: f1_score(y_test, (y_proba_dl >= t).astype(int), pos_label=1, zero_division=0),
)
print(f'\nBest threshold (by F1 kelas 1): {best_thr}')
y_pred_dl = (y_proba_dl >= best_thr).astype(int)
print(classification_report(y_test, y_pred_dl, digits=4))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_dl):.4f}')

In [ ]:
import seaborn as sns
cm = confusion_matrix(y_test, y_pred_dl)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=['No (0)', 'Yes (1)'], yticklabels=['No (0)', 'Yes (1)'])
ax.set_title(f'Confusion Matrix — DL @ threshold={best_thr}', fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dl_confusion_matrix.png', dpi=150)
plt.show()

## 10. Simpan Model (.keras — Format Produksi)

In [ ]:
model_path = MODELS_DIR / 'pulsevera_dl_model.keras'
model.save(model_path)
print(f'Model DL disimpan: {model_path} ({model_path.stat().st_size / 1024**2:.1f} MB)')

dl_metadata = {
    'framework': 'tensorflow',
    'tf_version': tf.__version__,
    'architecture': 'Dense[256-128-64-1] + BN + Dropout',
    'custom_components': ['focal_loss', 'EarlyStoppingByRecall'],
    'best_threshold': float(best_thr),
    'test_accuracy': float(accuracy_score(y_test, y_pred_dl)),
    'test_recall_pos': float(recall_score(y_test, y_pred_dl, pos_label=1, zero_division=0)),
    'test_precision_pos': float(precision_score(y_test, y_pred_dl, pos_label=1, zero_division=0)),
    'test_f1_pos': float(f1_score(y_test, y_pred_dl, pos_label=1, zero_division=0)),
    'test_roc_auc': float(roc_auc_score(y_test, y_proba_dl)),
    'class_weight': class_weight,
    'feature_order': FEATURE_ORDER,
    'tensorboard_log_dir': str(run_dir),
}
with open(MODELS_DIR / 'dl_metadata.json', 'w') as f:
    json.dump(dl_metadata, f, indent=2)
print(f'DL metadata: {MODELS_DIR / "dl_metadata.json"}')

## 11. Reload Test — pastikan custom_objects ter-register dengan baik

In [ ]:
reloaded = keras.models.load_model(
    model_path,
    custom_objects={'focal_loss': focal_loss},
    safe_mode=False,
)
sample = X_test_s[:5]
p_orig = model.predict(sample, verbose=0).ravel()
p_load = reloaded.predict(sample, verbose=0).ravel()
diff = np.abs(p_orig - p_load).max()
print(f'Predictions identical? max diff = {diff:.2e}  (harus mendekati 0)')

## 12. Ringkasan

| Item | Status |
|---|---|
| TensorFlow Functional API | ✓ |
| Custom Loss (Focal Loss) | ✓ |
| Custom Callback (EarlyStoppingByRecall) | ✓ |
| TensorBoard logging | ✓ |
| Class weighting | ✓ |
| Format `.keras` | ✓ |
| Reload + verifikasi prediksi | ✓ |

Lanjut: `07_evaluation_shap.ipynb` untuk evaluasi komprehensif + interpretabilitas SHAP.